# Limpieza de Nombres - Customers
Prueba de la función `clean_name` con datos reales

In [0]:
# Celda de configuración inicial
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.profile("ecommerce-etl-pipeline").getOrCreate()
    print("Sesión Spark creada")
except Exception as e:
    print(f"Usando spark existente: {e}")

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
print(f"Path: {sys.path[0]}")

In [0]:
# 1. Cargar datos crudos de bronze
storage_account_name = "stecommercedirty01"
container = "bronze"
path = f"abfss://{container}@{storage_account_name}.dfs.core.windows.net/customers/"

df_customers = spark.read.option("multiLine", True).json(path)
print(f"Filas: {df_customers.count()}")
df_customers.printSchema()

In [0]:
# 2. Ver los nombres sucios ANTES de limpiar
df_customers.select("name", "email", "city", "country").show(20, truncate=False)

In [0]:
# 3. Importar funciones de limpieza
from transformacion.cleaning import clean_name, clean_email

# Aplicar limpieza a name, city y email
df_limpio = clean_name(df_customers, "name")
df_limpio = clean_name(df_limpio, "city")
df_limpio = clean_email(df_limpio)

print("Columnas nuevas:", [c for c in df_limpio.columns if c.endswith("_clean")])

In [0]:
# 4. Comparar ANTES vs DESPUÉS de la limpieza
from pyspark.sql.functions import col

df_limpio.select(
    col("name"), col("name_clean"),
    col("email"), col("email_clean"),
    col("city"), col("city_clean")
).filter(col("name_clean") != col("name")) \
    .show(30, truncate=False)

In [0]:
# 5. Verificar caracteres especiales ELIMINADOS en name
from pyspark.sql.functions import regexp_extract

especiales = df_limpio.filter(
    regexp_extract(col("name_clean"), "[^a-zA-Z\\s'-]", 0) != ""
).select("name", "name_clean")

print(f"Nombres con caracteres especiales restantes: {especiales.count()}")
especiales.show(10)

In [0]:
# 6. Verificar emails inválidos → UNKNOWN
df_limpio.filter(col("email_clean") == "UNKNOWN") \
    .select("email", "email_clean") \
    .show(20, truncate=False)

print(f"Total emails UNKNOWN: {df_limpio.filter(col('email_clean') == 'UNKNOWN').count()}")

In [0]:
# 7. Vista final limpia
df_limpio.select(
    "id", "name_clean", "email_clean", "city_clean"
).show(30, truncate=False)